# 03 — Federated K-Means Runs

Run federated k-means via FeatureCloud and aggregate results.

**What this notebook does:**
1. Prepare per-site input directories for the FeatureCloud fc_kmeans app.
2. Launch federated k-means tests (requires Docker + FeatureCloud CLI).
3. OR aggregate existing federated outputs (if FeatureCloud is unavailable).
4. Join federated cluster assignments with metadata.

**Prerequisites:**
- Run notebook 01 first (data preparation).
- For live runs: Docker running + FeatureCloud CLI installed.
- For aggregation only: existing zip results in `real_datasets/<dataset>/inputs/*/tests/`.

## Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation_utils.real_datasets_utils import (
    dataset_configs,
    discover_clients,
    load_feature_matrix,
    load_metadata,
    prepare_variant_inputs,
    aggregate_fed_clusters,
)
from evaluation_utils.featurecloud_kmeans_utils import (
    ensure_app_image,
    run_single_federated_variant,
    aggregate_existing_federated_output,
)

## Configuration

In [ ]:
DATASETS = ["ecoli", "ovarian_cancer", "ccRCC_proteomics", "multiomics"]

# Set to True to launch a real FeatureCloud test.
# Set to False to only aggregate existing zip results.
RERUN_FEDERATED = True

# FeatureCloud settings (only needed when RERUN_FEDERATED=True)
APP_IMAGE = "fc_kmeans_upd"
APP_SOURCE_DIR = NOTEBOOK_DIR.parent / "federated_kmeans_upd"
CONTROLLER_HOST = "http://localhost:8000"
QUERY_INTERVAL = 5
TIMEOUT = 1800

OUTPUT_ROOT = NOTEBOOK_DIR

## Prepare FeatureCloud Inputs

Build per-site input directories with `intensities.tsv`, `design.tsv`,
and `config_kmeans.yml` — the format required by the fc_kmeans app.

This step creates `real_datasets/<dataset>/inputs/{before,corrected}/<site>/`
from the prepared matrices saved by notebook 01. It always runs (fast and idempotent)
so that the federated step below has up-to-date inputs.

In [ ]:
# Always generate per-site input directories (fast, idempotent).
# These are needed for both fresh federated runs and for aggregating existing results.
configs = dataset_configs(REPO_ROOT)

for ds_name in DATASETS:
    cfg = configs[ds_name]
    ds_dir = OUTPUT_ROOT / ds_name
    prepared_dir = ds_dir / "prepared"

    # Load prepared data from notebook 01
    before = load_feature_matrix(prepared_dir / "before_matrix.tsv")
    corrected = load_feature_matrix(prepared_dir / "corrected_matrix.tsv")
    metadata = pd.read_csv(prepared_dir / "metadata.tsv", sep="\t")
    clients = discover_clients(cfg)

    k_condition = int(metadata['condition'].nunique())
    k_batch = int(metadata['lab'].nunique())
    k_values = sorted(set([k_condition, k_batch]))

    print(f"\n{'='*60}")
    print(f"{ds_name}: preparing FC inputs for k={k_values}")
    print(f"{'='*60}")

    # Generate per-site input directories for the fc_kmeans app
    before_dir = prepare_variant_inputs(
        dataset_root=ds_dir, variant_name="before",
        matrix=before, metadata=metadata,
        clients=clients, k_values=k_values,
    )
    corrected_dir = prepare_variant_inputs(
        dataset_root=ds_dir, variant_name="corrected",
        matrix=corrected, metadata=metadata,
        clients=clients, k_values=k_values,
    )
    print(f"Created: {before_dir}")
    print(f"Created: {corrected_dir}")

## Run or Aggregate Federated K-Means

When `RERUN_FEDERATED=True`, this launches a FeatureCloud test for each variant.
When `False`, it aggregates existing zip results from previous runs.

In [ ]:
for ds_name in DATASETS:
    cfg = configs[ds_name]
    ds_dir = OUTPUT_ROOT / ds_name
    metadata = pd.read_csv(ds_dir / "prepared" / "metadata.tsv", sep="\t")
    clients = discover_clients(cfg)
    client_names = [c.name for c in clients]

    k_condition = int(metadata['condition'].nunique())
    k_batch = int(metadata['lab'].nunique())
    k_values = sorted(set([k_condition, k_batch]))

    print(f"\n{'='*60}")
    print(f"{ds_name}: {'running' if RERUN_FEDERATED else 'aggregating'} federated k-means")
    print(f"{'='*60}")

    for variant in ["before", "corrected"]:
        variant_input_dir = ds_dir / "inputs" / variant

        try:
            output_path = run_single_federated_variant(
                dataset_name=ds_name,
                variant_label=variant,
                variant_input_dir=variant_input_dir,
                dataset_root=ds_dir,
                metadata=metadata,
                client_names=client_names,
                k_values=k_values,
                app_image=APP_IMAGE,
                controller_host=CONTROLLER_HOST,
                query_interval=QUERY_INTERVAL,
                timeout=TIMEOUT,
                keep_extracted=False,
                aggregate_only=not RERUN_FEDERATED,
            )
            print(f"  {variant}: saved to {output_path}")
        except FileNotFoundError as e:
            print(f"  {variant}: SKIPPED — {e}")
        except RuntimeError as e:
            print(f"  {variant}: FAILED — {e}")

## Verify Outputs

Check that federated metadata files were produced.

In [ ]:
for ds_name in DATASETS:
    run_dir = OUTPUT_ROOT / ds_name / "kmeans_res" / "runs"
    for fname in ["1_metadata_before_fedclusters.tsv", "1_metadata_after_fedclusters.tsv"]:
        p = run_dir / fname
        if p.exists():
            df = pd.read_csv(p, sep="\t")
            print(f"{ds_name}/{fname}: {df.shape[0]} samples, cols={list(df.columns)}")
        else:
            print(f"{ds_name}/{fname}: NOT FOUND")